## 1. Імпорт бібліотек

In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print(f"PyTorch версія: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

## 2. Конфігурація

In [ ]:
FILE_PATH = "../../data/final_dataset.csv"
PATH_XLM_R = "../../data/xml-bert-results/checkpoint-1748"
PATH_MBERT = "../../data/results_mbert/checkpoint-2622"
PATH_UKRBERT = "../../data/results_ukrbert/checkpoint-2622"

TEXT_COLUMN = "text"
LABEL_COLUMN = "fake"
MAX_LENGTH = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Використовуємо пристрій: {device}")

## 3. Завантаження даних

In [ ]:
print(f"Завантаження даних з {FILE_PATH}...")
df = pd.read_csv(FILE_PATH, encoding='utf-8')

df = df.rename(columns={LABEL_COLUMN: 'label'})
df = df[[TEXT_COLUMN, 'label']]
df = df.dropna(subset=[TEXT_COLUMN, 'label'])

# Розділяємо: 70% train, 15% validation, 15% test
df_train_val, df_test = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
df_train, df_val = train_test_split(df_train_val, test_size=0.176, random_state=42, stratify=df_train_val['label'])

validation_dataset = Dataset.from_pandas(df_val)
test_dataset = Dataset.from_pandas(df_test)

print(f"Валідаційна вибірка: {len(validation_dataset)} зразків")
print(f"Тестова вибірка: {len(test_dataset)} зразків")

## 4. Завантаження моделей

In [ ]:
import os

def load_model_and_tokenizer(path):
    print(f"Завантаження моделі з {path}...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Директорія {path} не знайдена!")
    model = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(path)
    return model, tokenizer

model_xlm_r, tok_xlm_r = load_model_and_tokenizer(PATH_XLM_R)
model_mbert, tok_mbert = load_model_and_tokenizer(PATH_MBERT)
model_ukrbert, tok_ukrbert = load_model_and_tokenizer(PATH_UKRBERT)

print("\nВсі моделі завантажено")

## 5. Функція для отримання прогнозів

In [ ]:
def get_prediction(text, model, tokenizer):
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
    
    probabilities = softmax(outputs.logits, dim=-1)[0]
    return probabilities.cpu().numpy()

## 6. Отримання прогнозів на валідаційній вибірці

In [ ]:
print("Отримання прогнозів від трьох моделей для валідаційної вибірки...\n")

validation_probs_xlm = []
validation_probs_ukr = []
validation_probs_mbert = []
validation_labels = []

for example in tqdm(validation_dataset, desc="Валідаційна вибірка"):
    text = example[TEXT_COLUMN]
    label = example['label']
    
    probs_xlm = get_prediction(text, model_xlm_r, tok_xlm_r)
    probs_ukr = get_prediction(text, model_ukrbert, tok_ukrbert)
    probs_mbert = get_prediction(text, model_mbert, tok_mbert)
    
    validation_probs_xlm.append(probs_xlm)
    validation_probs_ukr.append(probs_ukr)
    validation_probs_mbert.append(probs_mbert)
    validation_labels.append(label)

validation_probs_xlm = np.array(validation_probs_xlm)
validation_probs_ukr = np.array(validation_probs_ukr)
validation_probs_mbert = np.array(validation_probs_mbert)
validation_labels = np.array(validation_labels)

print(f"\nПрогнози отримано для {len(validation_labels)} зразків")

## 7. Grid Search для оптимізації ваг

In [ ]:
from sklearn.metrics import fbeta_score

def grid_search_ensemble_weights(probs_xlm, probs_ukr, probs_mbert, labels, step=0.05):
    print("\n" + "="*70)
    print("Grid Search для оптимізації ваг ансамблю")
    print("="*70)
    print(f"Крок сітки: {step}\n")
    
    weight_range = np.arange(0, 1 + step, step)
    
    best_accuracy = 0
    best_f2 = 0
    best_recall = 0
    best_weights = None
    best_metrics = None
    all_results = []
    
    accuracy_metric = evaluate.load("accuracy")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    
    total_iterations = len(weight_range) ** 2
    
    with tqdm(total=total_iterations, desc="Grid Search") as pbar:
        for w_xlm in weight_range:
            for w_ukr in weight_range:
                w_mbert = 1.0 - w_xlm - w_ukr
                
                pbar.update(1)
                
                if w_mbert < -0.001 or w_mbert > 1.001:
                    continue
                
                w_mbert = round(w_mbert, 4)
                if w_mbert < 0 or w_mbert > 1:
                    continue
                
                weighted_probs = (
                    w_xlm * probs_xlm + 
                    w_ukr * probs_ukr + 
                    w_mbert * probs_mbert
                )
                
                predictions = np.argmax(weighted_probs, axis=1)
                
                accuracy = accuracy_metric.compute(predictions=predictions.tolist(), references=labels.tolist())['accuracy']
                precision = precision_metric.compute(predictions=predictions.tolist(), references=labels.tolist())['precision']
                recall = recall_metric.compute(predictions=predictions.tolist(), references=labels.tolist())['recall']
                f2 = fbeta_score(labels, predictions, beta=2)
                
                result = {
                    'w_xlm': round(w_xlm, 4),
                    'w_ukr': round(w_ukr, 4),
                    'w_mbert': round(w_mbert, 4),
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f2': f2
                }
                all_results.append(result)
                
                if (accuracy > best_accuracy or 
                    (accuracy == best_accuracy and f2 > best_f2) or
                    (accuracy == best_accuracy and f2 == best_f2 and recall > best_recall)):
                    best_accuracy = accuracy
                    best_f2 = f2
                    best_recall = recall
                    best_weights = (round(w_xlm, 4), round(w_ukr, 4), round(w_mbert, 4))
                    best_metrics = result.copy()
    
    print(f"\n{'='*70}")
    print("Оптимальні ваги:")
    print(f"  XLM-RoBERTa:  {best_weights[0]:.4f}")
    print(f"  ukr-roberta:  {best_weights[1]:.4f}")
    print(f"  mBERT:        {best_weights[2]:.4f}")
    print(f"\nМетрики на валідаційній вибірці:")
    print(f"  Accuracy:     {best_metrics['accuracy']:.4f}")
    print(f"  Precision:    {best_metrics['precision']:.4f}")
    print(f"  Recall:       {best_metrics['recall']:.4f}")
    print(f"  F2-Score:     {best_metrics['f2']:.4f}")
    print("="*70)
    
    return best_weights, best_metrics, all_results


best_weights, best_metrics, all_results = grid_search_ensemble_weights(
    validation_probs_xlm,
    validation_probs_ukr,
    validation_probs_mbert,
    validation_labels,
    step=0.05
)

## 8. Оцінка ансамблю на тестовій вибірці

In [ ]:
print("\nОтримання прогнозів від трьох моделей для тестової вибірки...\n")

all_predictions = []
all_labels = []

for example in tqdm(test_dataset, desc="Тестова вибірка"):
    text = example[TEXT_COLUMN]
    label = example['label']
    
    probs_xlm_r = get_prediction(text, model_xlm_r, tok_xlm_r)
    probs_mbert = get_prediction(text, model_mbert, tok_mbert)
    probs_ukrbert = get_prediction(text, model_ukrbert, tok_ukrbert)
    
    weighted_probs = (
        best_weights[0] * probs_xlm_r + 
        best_weights[1] * probs_ukrbert + 
        best_weights[2] * probs_mbert
    )
    
    final_prediction = np.argmax(weighted_probs)
    
    all_predictions.append(final_prediction)
    all_labels.append(label)

print("Прогнози ансамблю зібрано")

## 9. Фінальна оцінка

In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

accuracy = accuracy_metric.compute(predictions=all_predictions, references=all_labels)
precision = precision_metric.compute(predictions=all_predictions, references=all_labels)
recall = recall_metric.compute(predictions=all_predictions, references=all_labels)
f2 = fbeta_score(all_labels, all_predictions, beta=2)

print("\n" + "="*70)
print("Оцінка ансамблю на тестовій вибірці")
print("="*70)
print(f"Accuracy:  {accuracy['accuracy']:.4f}")
print(f"Precision: {precision['precision']:.4f}")
print(f"Recall:    {recall['recall']:.4f}")
print(f"F2-score:  {f2:.4f}")
print("="*70)